In [ ]:
from llama_cpp import Llama
import pandas as pd
import json
import re
from tqdm import tqdm
from sklearn.metrics import precision_score, recall_score, f1_score


# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "0"


llm = Llama.from_pretrained(
    repo_id="bartowski/JSL-MedLlama-3-8B-v2.0-GGUF",
    filename="JSL-MedLlama-3-8B-v2.0-Q4_K_M.gguf",  # you can pick other quants too

    n_gpu_layers=-1,
    n_threads=4,    
    n_batch=256,

    use_mmap=False,
    use_mlock=True
)

In [ ]:
def extract_yes_no_only(output_text):
    """
    Extract only Yes/No from the beginning of the LLM output.
    """

    # Strip whitespace
    text = output_text.strip()

    # Split by space to get first word
    first_word = text.split()[0].lower()

    # Normalize
    if first_word.startswith("yes"):
        return "Yes"
    if first_word.startswith("no"):
        return "No"

    # Fallback if something unexpected
    return "Unknown"

In [ ]:

def extract_drug_adr(text):
    prompt = f"""
    You are an excellent clinical information classifier system. Decide whether the text describes an Adverse Drug Effect.  
    Respond strictly with "Yes" or "No". An adverse drug event is any harmful or negative experience related to ongoing medication or treatment. 

    Text: {text}

    Answer:
    """

    result = llm(
        prompt=prompt,
        max_tokens=256,
        temperature=0.0
    )

    raw_output = result["choices"][0]["text"].strip()
    cleaned_op = extract_yes_no_only(raw_output)
    print("Output:",cleaned_op, "##END")
    return cleaned_op

input_csv_path = "./data/LLaVA-Med/All_ADRs - Sheet1.csv"
output_csv_path = "./data/LLaVA-Med/Results_MedLLaMA1_ClassificationADR_151125_1.csv" 


df = pd.read_csv(input_csv_path)
print(df.columns.tolist())

texts = df["Preprocessed Posts"].fillna("").astype(str).tolist()
gold_labels = df["Adverse effects(Yes/No)"].astype(str).str.strip()

In [ ]:
# ------------------------------
# 4. Run Classification
# ------------------------------
predictions = []

for text in texts:
    pred = extract_drug_adr(str(text))
    print(pred)
    predictions.append(pred)

df["Output - Adverse effects(Yes/No)"] = predictions


# ------------------------------
# 5. Compute Metrics
# ------------------------------
prec = precision_score(gold_labels, predictions, pos_label="Yes")
rec = recall_score(gold_labels, predictions, pos_label="Yes")
f1 = f1_score(gold_labels, predictions, pos_label="Yes")

df["Cumulative Precision"] = ""
df["Cumulative Recall"] = ""
df["Cumulative F1 Score"] = ""

df.loc[df.index[0], "Cumulative Precision"] = prec
df.loc[df.index[0], "Cumulative Recall"] = rec
df.loc[df.index[0], "Cumulative F1 Score"] = f1


# ------------------------------
# 6. Save Output CSV
# ------------------------------
df.to_csv(output_csv_path, index=False)

print(f"\nDone! File saved: {output_csv_path}")
print("Precision:", prec)
print("Recall:", rec)
print("F1 Score:", f1)